<a href="https://colab.research.google.com/github/Sixcharcoin/IntermediateDataScience_Projects/blob/main/lecture03_homework_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第3回講義 宿題

## 課題

今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) を多層パーセプトロンによって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値

Accuracy 85%

### ルール

- 訓練データは`x_train`， `t_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train`，`t_train`以外の学習データは使わないでください．**
- **多層パーセプトロンのアルゴリズム部分は第3回の演習を参考に，NumPyのみで実装してください．** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第3回 ニューラルネットワーク基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第3回 ニューラルネットワーク基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 作業ディレクトリを指定
work_dir = '/content/drive/MyDrive/Colab Notebooks/DLbasic'

### データの読み込み（このセルは修正しないでください）

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import inspect


#学習データ
x_train = np.load(work_dir + '/Lecture03/data/x_train.npy')
t_train = np.load(work_dir + '/Lecture03/data/y_train.npy')

#テストデータ
x_test = np.load(work_dir + '/Lecture03/data/x_test.npy')

# データの前処理（正規化， one-hot encoding)
x_train, x_test = x_train / 255., x_test / 255.
x_train, x_test = x_train.reshape(x_train.shape[0], -1), x_test.reshape(x_test.shape[0], -1)
t_train = np.eye(N=10)[t_train.astype("int32").flatten()]

### 多層パーセプトロンの実装

In [6]:
# データの分割
x_train, x_val, t_train, t_val =\
    train_test_split(x_train, t_train, test_size=10000)

In [7]:
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))


def create_batch(data, batch_size):#ミニバッチか
    """
    :param data: np.ndarray，入力データ
    :param batch_size: int，バッチサイズ
    """
    num_batches, mod = divmod(data.shape[0], batch_size)
    batched_data = np.split(data[: batch_size * num_batches], num_batches)
    if mod:
        batched_data.append(data[batch_size * num_batches:])

    return batched_data

In [14]:
# シード値を変えることで何が起きるかも確かめてみてください．
rng = np.random.RandomState(1234)
random_state = 42


# 発展: 今回の講義で扱っていない活性化関数について調べ，実装してみましょう
def relu(x):
  return np.maximum(0, x)

def deriv_relu(x):
  return (x > 0).astype(int)

def softmax(x):
    x -= x.max(axis=1, keepdims=True)
    x_exp = np.exp(x)
    return x_exp / np.sum(x_exp, axis=1, keepdims=True)

def deriv_softmax(x):
    return softmax(x) * (1 - softmax(x))

def crossentropy_loss(t, y):
  return -np.mean(np.sum(t * np.log(np.clip(y, 1e-10, 1)), axis=1))
   #これ二値用らしいreturn (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean()

class Dense:
  #AdamWを追加
    def __init__(self, in_dimention, out_dimention, function, deriv_function):
      self.W = np.random.uniform(low = -0.08, high = 0.08, size = (in_dimention, out_dimention)).astype('float64')
      self.b = np.zeros(out_dimention).astype('float64')
      self.function = function
      self.deriv_function = deriv_function
      self.x = None
      self.u = None
      self.dW = None
      self.db = None
      self.params_idxs = np.cumsum([self.W.size, self.b.size]) #パラメーターを一緒くたにした時にどこまでがWでどこからがbかをしめす
      self.moment_W = np.zeros_like(self.W)
      self.v_W = np.zeros_like(self.W)
      self.moment_b = np.zeros_like(self.b)
      self.v_b = np.zeros_like(self.b)
      self.t = 0




    def __call__(self, x): #順伝播
      self.x = x
      self.u = np.matmul(self.x, self.W) + self.b
      h = self.function(self.u)
      return h

    def back_propagation(self, delta, W):
        #逆伝播
      self.delta = self.deriv_function(self.u) * np.matmul(delta, W.T)
      return self.delta

    def compute_gradient(self):
      batch_size = self.delta.shape[0]
      self.dW = np.matmul(self.x.T, self.delta) / batch_size
      self.db = np.matmul(np.ones(batch_size), self.delta) / batch_size

    def get_parametres(self):
      return np.concatenate([self.W.ravel(), self.b], axis=0)

    def set_parametres(self, parametres):
      _W, _b = np.split(parametres, self.params_idxs)[:-1]
      self.W = _W.reshape(self.W.shape)
      self.b = _b

    def get_gradients(self):
      return np.concatenate([self.dW.ravel(), self.db], axis=0)


class Model:
    def __init__(self, hidden_dimentions, activation_functions, deriv_functions):
      self.layers = []
      for i in range(len(hidden_dimentions)-2):
        #出力層以外をレイヤーにアペン
        self.layers.append(Dense(hidden_dimentions[i], hidden_dimentions[i+1], activation_functions[i], deriv_functions[i]))
      self.layers.append(Dense(hidden_dimentions[-2], hidden_dimentions[-1], activation_functions[-1], deriv_functions[-1]))#出力層

    def __call__(self, x):
      return self.forward(x)

    def forward(self, x):
      for layer in self.layers:
        x = layer(x)
      return x

    def backward(self, delta):
      batch_size = delta.shape[0]

      for i, layer in enumerate(self.layers[::-1]):
        if i == 0:
          layer.delta = delta
          layer.compute_gradient()
        else:
          delta = layer.back_propagation(delta, W)
          layer.compute_gradient()

        W = layer.W

    def update(self, lr, beta1=0.9, beta2=0.999, weight_decay=0.01, eps=1e-8):
      for layer in self.layers:
        layer.t += 1
        # 重みWのAdamW更新
        layer.moment_W = beta1 * layer.moment_W + (1 - beta1) * layer.dW
        layer.v_W = beta2 * layer.v_W + (1 - beta2) * (layer.dW ** 2)
        m_hat_W = layer.moment_W / (1 - beta1 ** layer.t)
        v_hat_W = layer.v_W / (1 - beta2 ** layer.t)
        adam_update_W = lr * m_hat_W / (np.sqrt(v_hat_W) + eps)
        layer.W -= adam_update_W  # Adam更新
        layer.W *= (1 - lr * weight_decay)  # AdamWの分離weight decay
        #layer.W -= lr * layer.dW
        #layer.b -= lr * layer.db

lr = 0.001#eta - learningrate
n_epochs = 100
batch_size = 128
#model
mlp = Model(hidden_dimentions=[784, 1024, 512, 10], activation_functions=[relu, relu, softmax], deriv_functions=[deriv_relu, deriv_relu, deriv_softmax])

### モデルの学習

In [15]:
def train_model(mlp, x_train, t_train, x_val, t_val, n_epochs=10):
    for epoch in range(n_epochs):
        losses_train = []
        losses_valid = []
        train_num = 0
        train_true_num = 0
        valid_num = 0
        valid_true_num = 0

        x_train, t_train = shuffle(x_train, t_train)
        x_train_batches, t_train_batches = create_batch(x_train, batch_size), create_batch(t_train, batch_size)

        x_val, t_val = shuffle(x_val, t_val)
        x_val_batches, t_val_batches = create_batch(x_val, batch_size), create_batch(t_val, batch_size)

        # モデルの訓練
        for x, t in zip(x_train_batches, t_train_batches):
            # 順伝播
            # WRITE ME
            y = mlp(x)
            # 損失の計算
            # WRITE ME
            loss = (-t * np_log(y)).sum(axis=1).mean()
            losses_train.append(loss.tolist())

            #多分逆伝播が抜けてると思うのでそれも書く
            delta = y - t
            mlp.backward(delta)

            # パラメータの更新
            # WRITE ME
            mlp.update(lr)

            # 精度を計算
            acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1), normalize=False)
            train_num += x.shape[0]
            train_true_num += acc

            #return cost, acc

        # モデルの評価
        for x, t in zip(x_val_batches, t_val_batches):
            # 順伝播
            # WRITE ME
            y = mlp(x)

            # 損失の計算
            # WRITE ME
            loss = (-t * np_log(y)).sum(axis=1).mean()
            losses_valid.append(loss.tolist())

            acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1), normalize=False)
            valid_num += x.shape[0]
            valid_true_num += acc


        #if epoch % 50 == 0 or epoch == n_epochs - 1:
        print('EPOCH: {}, Train [Loss: {:.3f}, Accuracy: {:.3f}], Valid [Loss: {:.3f}, Accuracy: {:.3f}]'.format(
            epoch,
            np.mean(losses_train),
            train_true_num/train_num,
            np.mean(losses_valid),
            valid_true_num/valid_num
        ))


train_model(mlp, x_train, t_train, x_val, t_val, n_epochs)

EPOCH: 0, Train [Loss: 0.484, Accuracy: 0.828], Valid [Loss: 0.398, Accuracy: 0.853]
EPOCH: 50, Train [Loss: 0.053, Accuracy: 0.981], Valid [Loss: 0.531, Accuracy: 0.895]
EPOCH: 99, Train [Loss: 0.040, Accuracy: 0.987], Valid [Loss: 0.679, Accuracy: 0.898]


In [16]:
t_pred = []
for x in x_test:
    # 順伝播
    x = x[np.newaxis, :]
    y = mlp(x)

    # モデルの出力を予測値のスカラーに変換
    pred = y.argmax(1).tolist()

    t_pred.extend(pred)

submission = pd.Series(t_pred, name='label')
submission.to_csv(work_dir + '/Lecture03/6_adam_submission_pred_03.csv', header=True, index_label='id')